In [ ]:
import mlflow
import pandas as pd
from dotenv import load_dotenv
from mlflow import MlflowClient
from mlflow.deployments import get_deploy_client

load_dotenv("../config/.env")
mlflow.set_registry_uri("databricks-uc")

#### Model Deployment

In [ ]:
client_mlflow = MlflowClient()
client_databricks = get_deploy_client("databricks")

In [ ]:
model_version = client_mlflow.get_model_version_by_alias(
    "betsim.models.xgboost_optuna",
    "champion",
)
version = model_version.version

In [ ]:
endpoint = client_databricks.create_endpoint(
    config={
        "name": "betsim",
        "config": {
            "served_entities": [
                {
                    "entity_name": "betsim.models.xgboost_optuna",
                    "entity_version": version,
                    "workload_size": "Small",
                    "workload_type": "CPU",
                    "scale_to_zero_enabled": True,
                },
            ],
        },
    },
)

In [ ]:
ep = client_databricks.get_endpoint("betsim")
ep["state"]["ready"]

#### Model Inference

In [ ]:
df = pd.read_parquet("../data/processed/jleague.parquet")
df.query("fold == 'development'", inplace=True)

data = df.head(3).drop(columns=["fold", "hcap_res"]).to_dict(orient="split")

In [ ]:
client_databricks.predict(
    inputs={"dataframe_split": data},
    endpoint="betsim",
)